# 15 · Load test, PNG check and registry (Step 6.1, D-065)

**Runtime: L4 GPU** (Runtime → Change runtime type → **L4 GPU**). The last Colab run of the project.

1. **One model server** (vLLM) with four adapters loaded: E5 and the three specialists.
2. **Load test with Locust:** 1, 2, 4, 8, 16 and 32 users sending real val photos back to back, 60 s per level (~30 min):
   - E5 in each cascade mode: `sequential`, `parallel_checks`, `speculative`;
   - the three specialists (one adapter per stage) in `parallel_checks`, to compare with E5.
3. **PNG check (~12 min):** the val replay again with lossless photos; does agreement with the offline evaluation improve? (D-063)
4. **Registry (~5 min):** E5 is uploaded to W&B as a versioned adapter, and the gateway starts from the registry: it finds the `production` version, downloads it and loads it into the running server. Needs the `WANDB_API_KEY` secret; skipped without it.

Results are small JSON files (numbers only). If Colab disconnects, **Run all** again: finished parts are skipped.

Every command runs through `sh()`, which stops with an error if the command fails.

## 1 · Settings, Drive, code

In [ ]:
import json
import os
import random
import secrets
import subprocess
import time
import urllib.error
import urllib.request

from google.colab import drive, userdata

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
LOCAL = "/content/data"
CKPT = f"{DRIVE_ROOT}/checkpoints"
RUNS = f"{DRIVE_ROOT}/runs"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"
LOCUST = "/content/locust-venv/bin/locust"
MODEL = "Qwen/Qwen3-VL-4B-Instruct"
ADAPTERS = {
    "e5-multi-lora": f"{CKPT}/e5-multi-lora/adapter",
    **{
        f"e2-{t}-lora": f"{CKPT}/e2-{t}-lora/adapter"
        for t in ("answer", "answerability", "quality")
    },
}
OUT = f"{RUNS}/load-test"
MODEL_URL, GATEWAY_URL = "http://127.0.0.1:8000", "http://127.0.0.1:8090"  # Colab itself uses 8080


def sh(cmd, check=True, **env):
    """Run a shell command, stream its output, and raise if it fails (unless check=False)."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"Command failed (exit {code}). Stop here and paste the output above.")
    return code


def gpu_used_mib():
    out = subprocess.run(
        "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
        shell=True,
        capture_output=True,
        text=True,
    ).stdout
    return int(out.split()[0])


def stop(pattern):
    sh(f'pkill -f "{pattern}"; sleep 2', check=False)


def stop_servers():
    """Stop model servers and gateways left by an earlier run; wait for the GPU."""
    stop("[v]llm serve")
    stop("[v]izwiz.serve.gateway")
    for _ in range(60):
        if gpu_used_mib() < 2000:
            break
        time.sleep(2)
    print(f"GPU memory in use: {gpu_used_mib()} MiB")


def get(url, timeout=5):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.status, json.loads(r.read())


def wait_for(url, seconds, proc=None, log=None):
    t0 = time.time()
    while time.time() - t0 < seconds:
        try:
            return get(url)[1], time.time() - t0
        except Exception:
            if proc is not None and proc.poll() is not None:
                break
            time.sleep(1)
    if log:
        sh(f"tail -n 40 {log}", check=False)
    raise RuntimeError(f"{url} did not come up. Paste the log lines above.")


os.environ["HF_HOME"] = "/content/hf_cache"
try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")  # never printed
    HAVE_WANDB = True
except Exception:
    HAVE_WANDB = False
print(
    "W&B key:",
    "loaded from Colab Secrets"
    if HAVE_WANDB
    else "not available: the registry part will be skipped",
)
# A folder created under /content/drive before mounting blocks the mount: remove empty ones only.
if not os.path.ismount("/content/drive"):
    subprocess.run("find /content/drive -depth -type d -empty -delete", shell=True, check=False)
drive.mount("/content/drive")
os.makedirs(OUT, exist_ok=True)  # only after Drive is mounted
for name, path in ADAPTERS.items():
    assert os.path.exists(f"{path}/adapter_config.json"), f"{name} adapter not found on Drive"
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
stop_servers()

**Paste back:** the commit line, the `W&B key` line and the GPU line (must say NVIDIA L4).

## 2 · Environments (~5 min)
The pinned model-server environment, plus Locust in its own small environment so it cannot change the pinned packages.

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e '{REPO_DIR}[serve]' -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")
sh("uv venv -q --clear --python 3.13 /content/locust-venv")
sh("uv pip install -q --python /content/locust-venv/bin/python locust==2.47.0")
sh(f"{LOCUST} --version")

## 3 · Unzip val photos (~4 min) and pick the load-test sample
300 val questions, fixed seed. Their photos are read into memory by the load generator; nothing about them is printed.

In [ ]:
%%time
import hashlib
from pathlib import Path

expected = json.load(open(f"{REPO_DIR}/results/splits/files.json"))
digest = hashlib.sha256(open(f"{DATA}/processed/vqa_val.jsonl", "rb").read()).hexdigest()
assert digest == expected["vqa_val.jsonl"]["sha256"], (
    "vqa_val.jsonl differs from the committed hash"
)
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits train")
paths = {p.name: str(p) for p in Path(f"{LOCAL}/images").rglob("*.jpg")}
recs = [json.loads(line) for line in open(f"{DATA}/processed/vqa_val.jsonl")]
sample = random.Random(0).sample(recs, 300)
json.dump(
    [{"path": paths[r["image"]], "question": r["question"]} for r in sample],
    open("/content/bench_samples.json", "w"),
)
sizes = sorted(os.path.getsize(paths[r["image"]]) for r in sample)
print(
    f"LOAD SAMPLE: 300 val questions; photo size median {sizes[150] / 1e6:.2f} MB, max {sizes[-1] / 1e6:.2f} MB"
)

**Paste back:** the `LOAD SAMPLE` line.

## 4 · Start the model server with four adapters (~3 min)
Runtime adapter loading is switched on for the registry part (`VLLM_ALLOW_RUNTIME_LORA_UPDATING`).

In [ ]:
%%time
stop_servers()
env = {
    **os.environ,
    "PATH": f"{VENV}/bin:" + os.environ["PATH"],
    "VLLM_USE_FLASHINFER_SAMPLER": "0",  # R-002, D-020
    "VLLM_ALLOW_RUNTIME_LORA_UPDATING": "True",
}
modules = " ".join(f"{n}={p}" for n, p in ADAPTERS.items())
vllm_cmd = (
    f"{VENV}/bin/vllm serve {MODEL} --dtype bfloat16 --max-model-len 4096 --gpu-memory-utilization 0.85 "
    f"--enable-lora --lora-modules {modules} --max-lora-rank 16 --max-loras 4 --max-cpu-loras 8 "
    f"--limit-mm-per-prompt '{{\"image\": 1}}' --generation-config vllm --seed 0 --port 8000"
)
vllm_proc = subprocess.Popen(
    vllm_cmd, shell=True, env=env, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT
)
models, secs = wait_for(f"{MODEL_URL}/v1/models", 900, vllm_proc, "/content/vllm.log")
print(f"model server up after {secs:.0f} s; models:", [m["id"] for m in models["data"]])
mem_lines = (
    subprocess.run(
        "grep -E 'Model loading took|Available KV cache memory|GPU KV cache size' /content/vllm.log | sed 's/^.*\\] //'",
        shell=True,
        capture_output=True,
        text=True,
    )
    .stdout.strip()
    .splitlines()
)
for line in mem_lines:
    print("MEM:", line)
server_info = {
    "gpu_mem_used_mib_idle": gpu_used_mib(),
    "vllm_memory": mem_lines,
    "adapters_loaded": list(ADAPTERS),
    "flags": "--max-model-len 4096 --gpu-memory-utilization 0.85 --max-loras 4 --max-lora-rank 16",
}
print("GPU memory in use with the server idle:", server_info["gpu_mem_used_mib_idle"], "MiB")

**Paste back:** the `model server up` line, the `MEM` lines and the GPU memory line.

## 5 · Load test (~30 min)
For each configuration the gateway is started with its own settings file, then Locust runs 10 s of warm-up and 60 s of measurement per user count. Rows are saved as they finish; a rerun skips finished configurations.

In [ ]:
%%time
import yaml

base_cfg = yaml.safe_load(open(f"{REPO_DIR}/configs/serve.yaml"))
E5 = {s: "e5-multi-lora" for s in ("quality", "answerability", "answer")}
SPEC = {s: f"e2-{s}-lora" for s in ("quality", "answerability", "answer")}
RUNS_TO_DO = [
    ("e5-sequential", E5, "sequential"),
    ("e5-parallel_checks", E5, "parallel_checks"),
    ("e5-speculative", E5, "speculative"),
    ("specialists-parallel_checks", SPEC, "parallel_checks"),
]
SWEEP = f"{OUT}/sweep.jsonl"
done = {json.loads(line)["label"] for line in open(SWEEP)} if os.path.exists(SWEEP) else set()


def start_gateway(cfg, name, extra_env=None):
    path = f"/content/serve-{name}.yaml"
    yaml.safe_dump(cfg, open(path, "w"))
    stop("[v]izwiz.serve.gateway")
    gw_env = {
        **env,
        "VIZWIZ_MODEL_URL": MODEL_URL,
        "VIZWIZ_SERVE_CONFIG": path,
        **(extra_env or {}),
    }
    proc = subprocess.Popen(
        f"{VENV}/bin/uvicorn --factory vizwiz.serve.gateway:create_app --host 127.0.0.1 --port 8090",
        shell=True,
        env=gw_env,
        cwd=REPO_DIR,
        stdout=open(f"/content/gateway-{name}.log", "w"),
        stderr=subprocess.STDOUT,
    )
    health, secs = wait_for(f"{GATEWAY_URL}/health", 300, proc, f"/content/gateway-{name}.log")
    return proc, health, secs


for label, adapters, mode in RUNS_TO_DO:
    if label in done:
        print(f"{label}: already done, skipped")
        continue
    proc, health, _ = start_gateway({**base_cfg, "adapters": adapters, "mode": mode}, label)
    print(f"{label}: gateway up, missing adapters {health['missing_adapters']}")
    sh(
        f"{PY} -m vizwiz.bench.sweep --label {label} --gateway {GATEWAY_URL} --model-server {MODEL_URL} --locust {LOCUST} "
        f"--samples /content/bench_samples.json --users 1 2 4 8 16 32 --seconds 60 --warmup 10 --out {SWEEP}"
    )
stop("[v]izwiz.serve.gateway")

**Paste back:** all `SWEEP` lines (24 of them).

## 6 · PNG check (~12 min)
The val replay of notebook 13 again, with lossless photos, compared with the same offline predictions.

In [ ]:
%%time
png_cfg = "/content/serve-png.yaml"
yaml.safe_dump({**base_cfg, "image_format": "png"}, open(png_cfg, "w"))
sh(
    f"{PY} -m vizwiz.serve.replay --records {DATA}/processed --images {LOCAL}/images --split val --model-url {MODEL_URL} "
    f"--config {png_cfg} --concurrency 16 --out {OUT}/replay_val_png.jsonl"
)
sh(
    f"{PY} -m vizwiz.serve.tune --records {DATA}/processed --val-replay {OUT}/replay_val_png.jsonl "
    f"--offline-val {RUNS}/e5-multi-lora-val --report {OUT}/tune_report_png.json"
)
par = json.load(open(f"{OUT}/tune_report_png.json"))["parity_val"]
print("PNG ANSWER:", {k: par["answer"][k] for k in ("same_answer_pct", "vqa_accuracy")})
print(
    "PNG ANSWERABILITY:",
    {k: par["answerability"][k] for k in ("same_decision_at_0.5_pct", "average_precision")},
)
print("PNG QUALITY:", {k: par["quality"][k] for k in ("same_labels_pct", "macro_f1")})

**Paste back:** the three `PNG` lines.

## 7 · Registry: upload E5 to W&B, start the gateway from it (~5 min)
Skipped without the W&B key. The admin token for promote/rollback is random for this session and never printed.

In [ ]:
%%time
reg_report = {"ran": False}
if not HAVE_WANDB:
    print("REGISTRY: skipped (no WANDB_API_KEY in Colab Secrets)")
else:
    sh(
        f"cd {REPO_DIR} && {PY} -m vizwiz.serve.registry register --name e5-multi-lora --path {ADAPTERS['e5-multi-lora']} "
        f"--task multi --run-id e5-multi-lora --metrics {REPO_DIR}/results/eval/e5-multi-lora/metrics.json --alias production"
    )
    token = secrets.token_hex(16)
    reg_cfg = {
        **base_cfg,
        "adapters": E5,
        "registry": {
            **base_cfg["registry"],
            "enabled": True,
            "cache_dir": "/content/adapter-cache",
        },
    }
    sh("rm -rf /content/adapter-cache", check=False)
    t0 = time.time()
    proc, health, secs = start_gateway(reg_cfg, "registry", {"VIZWIZ_ADMIN_TOKEN": token})
    startup_s = time.time() - t0
    adapters_now = get(f"{GATEWAY_URL}/adapters")[1]
    print(
        "REGISTRY: gateway up after",
        round(startup_s, 1),
        "s (resolve + download + load); stages:",
        adapters_now["stages"],
    )
    print("REGISTRY live:", {n: v["version"] for n, v in adapters_now["registry"]["live"].items()})

    def admin(path):
        req = urllib.request.Request(
            f"{GATEWAY_URL}{path}",
            method="POST",
            data=b"{}",
            headers={"X-Admin-Token": token, "Content-Type": "application/json"},
        )
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                return r.status, json.loads(r.read())
        except urllib.error.HTTPError as e:
            return e.code, json.loads(e.read())

    rb = admin("/adapters/e5-multi-lora/rollback")
    print("REGISTRY rollback with no previous version (expect 409):", rb[0])
    statuses = []
    for s in json.load(open("/content/bench_samples.json"))[:5]:
        b = subprocess.run(
            [
                "curl",
                "-s",
                "-F",
                f"image=@{s['path']}",
                "-F",
                f"question={s['question']}",
                f"{GATEWAY_URL}/v1/assist",
            ],
            capture_output=True,
            text=True,
        ).stdout
        statuses.append(json.loads(b)["status"])
    print("REGISTRY requests through the registry-loaded adapter:", statuses)
    reg_report = {
        "ran": True,
        "startup_s": round(startup_s, 1),
        "stages": adapters_now["stages"],
        "live": {n: v["version"] for n, v in adapters_now["registry"]["live"].items()},
        "rollback_without_previous_status": rb[0],
        "requests_ok": len(statuses),
    }
    stop("[v]izwiz.serve.gateway")
json.dump(reg_report, open(f"{OUT}/registry_check.json", "w"), indent=2)

**Paste back:** all `REGISTRY` lines.

## 8 · Save, stop, disconnect
Upload back (numbers only):
- `MyDrive/vizwiz/runs/load-test/sweep.jsonl`
- `MyDrive/vizwiz/runs/load-test/server_info.json`
- `MyDrive/vizwiz/runs/load-test/tune_report_png.json`
- `MyDrive/vizwiz/runs/load-test/registry_check.json`

Then Runtime → **Disconnect and delete runtime**.

In [ ]:
server_info["versions"] = subprocess.run(
    f"{PY} -m vizwiz.compat versions", shell=True, capture_output=True, text=True
).stdout.strip()
server_info["locust"] = subprocess.run(
    f"{LOCUST} --version", shell=True, capture_output=True, text=True
).stdout.strip()
server_info["git_commit"] = subprocess.run(
    f"git -C {REPO_DIR} rev-parse --short HEAD", shell=True, capture_output=True, text=True
).stdout.strip()
json.dump(server_info, open(f"{OUT}/server_info.json", "w"), indent=2)
stop_servers()
drive.flush_and_unmount()
print("Saved, servers stopped, Drive flushed. Safe to disconnect.")